# Fig 2E — Sketch dimension and fine-mapping accuracy (data processing)

Power / FDR of SuSiE-RSS and SuSiE-rRSS with sketch- or subset-based LD across sketch dimension B, against in-sample LD (UK Biobank). Bootstrap 95% CIs over replicates.

**Output:** `figure_data/Fig2B.Finemap_Sketch_Dim.rds` (+ `.csv`)

In [7]:
source("../../R/utils.R")

data_name <- "Fig2B.Finemap_Sketch_Dim"

res <- readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKBB_B_finemap/summary/result_susie_cs_summary_summary.rds")

## ---- method labels (palette pal_sketch_dim in R/utils.R) ---------------------
## order: insample, SuSiE-rRSS (sketch, subset), SuSiE-RSS (sketch, subset)
method_lv <- names(pal_sketch_dim)

recode_method <- c(
  insample   = "SuSiE-RSS (In-sample LD)",
  sketch_mix = "SuSiE-rRSS (sketch)",
  subset_mix = "SuSiE-rRSS (subset)",
  sketch     = "SuSiE-RSS (sketch)",
  subset     = "SuSiE-RSS (subset)"
)

In [8]:
## ============================================================================
## pooled metrics + bootstrap 95% CI
## resample the replicate rows within each cell, recompute the pooled metric
## ============================================================================
set.seed(1); B_boot <- 1000

boot_cell_dim <- function(d, key, Bb = B_boot) {
  n   <- nrow(d)
  pow <- sum(d$true_var_number) / (n * key$total_var_number)
  fdr <- if (sum(d$total_set_number) == 0) 0 else 1 - sum(d$true_set_number) / sum(d$total_set_number)
  bp <- bf <- numeric(Bb)
  for (b in seq_len(Bb)) {
    dd <- d[sample.int(n, n, replace = TRUE), ]
    bp[b] <- sum(dd$true_var_number) / (n * key$total_var_number)
    bf[b] <- if (sum(dd$total_set_number) == 0) 0 else 1 - sum(dd$true_set_number) / sum(dd$total_set_number)
  }
  tibble(
    power = pow, power_lo = quantile(bp, .025), power_hi = quantile(bp, .975),
    FDR   = fdr, FDR_lo   = quantile(bf, .025), FDR_hi   = quantile(bf, .975),
    ave_size = sum(d$total_set_length) / sum(d$total_set_number),
    f1_score = if_else(pow == 0 & fdr == 1, 0, pow * (1 - fdr) / (pow + 1 - fdr)),
    rep = n
  )
}

summ <- res %>%
  group_by(method, h2g, B_stoc, total_var_number) %>%
  group_modify(~ boot_cell_dim(.x, .y)) %>%
  ungroup()

In [9]:
## ---- plot-ready data: NCP facets, B facets, ordered method factor -----------
n_gwas <- 300000L    ## <-- SET UKBB sample size; 800 is GTEx eQTL, NOT UKBB. NCP = n * h2g

unmapped <- setdiff(unique(as.character(summ$method)), names(recode_method))
if (length(unmapped)) warning("Unmapped methods dropped: ", paste(unmapped, collapse = ", "))

plot_dt <- summ %>%
  mutate(
    B_stoc  = as.integer(str_extract(as.character(B_stoc), "[0-9]+")),
    h2g_num = as.numeric(str_remove(as.character(h2g), "h2g_?")),
    ncp_num = n_gwas * h2g_num,
    ncp     = fct_reorder(paste0("NCP = ", formatC(ncp_num, format = "f", digits = 1)), ncp_num),
    B_lab   = factor(paste0("Sketch Dim = ", B_stoc), levels = paste0("Sketch Dim = ", sort(unique(B_stoc)))),
    method  = recode(method, !!!recode_method),
    method  = factor(method, levels = method_lv),
    varnum  = factor(total_var_number)
  ) %>%
  filter(!is.na(method)) %>%
  filter(B_stoc %in% c(500, 1000, 5000, 10000, 20000))   # sketch dimensions shown in the figure

save_data(plot_dt, data_name)